# 3) Chains
# Combine LLMs and Prompt templates to build workflows

# this method is called LCEL (LangChain Expression Language)
# Connect AI components such as prompts, models, data retrievers, and parsers using a simple, chainable syntax with the pipe | operator, enabling smooth data flow from one component to the next.
# It is designed to simplify the creation of complex AI pipelines while keeping code clean, modular, and readable.

# Runnable
# -------
# A LangChain Runnable is any component that can receive an input, perform an operation, and return an output.
# Prompt templates, LLMs, output parsers and retrievers are examples of runnables.

In [ ]:
import os
from dotenv import load_dotenv
import warnings
from google import genai
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
# from langchain_core.messages import AIMessage

In [ ]:
warnings.filterwarnings("ignore")

env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# i) Create the Gemini client
client = genai.Client(api_key=gemini_key)

In [ ]:
# ii) Create a function that calls Gemini
def call_gemini(prompt_value):
    # Convert LangChain's PromptValue into a string
    prompt_text = prompt_value.to_string()
    response = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt_text)
    return response.text

In [ ]:
# iii) Create a reusable prompt template
prompt = PromptTemplate.from_template(
    """
    A company is planning to conduct a training program on {topic}
    for its {audience}.

    Suggest a suitable program name in about {word_count} words.
    Return only the program name.
    """
    )

In [ ]:
# iv) Convert the Gemini function into a LangChain runnable
gemini_runnable = RunnableLambda(call_gemini)

# v) Create the chain
chain = prompt | gemini_runnable

In [ ]:
# vi) Invoke the chain
result = chain.invoke({ "topic": "Artificial Intelligence", "audience": "senior leaders", "word_count": 5})
print(result)

In [ ]:
result = chain.invoke({"topic": "Generative AI for All", "audience":"all", "word_count":7})
print(result)

In [ ]:
# Example 2
# Training prompt -> Gemini runnable -> String output parser -> Final Output

gemini_runnable = RunnableLambda(call_gemini)
output_parser = StrOutputParser()

# Prompt 1
prompt1 = PromptTemplate.from_template(
    """
    Explain the basic rules of {sport}.
    Give the answer in three short sentences.
    """
)

# Create the chain
sports_chain = prompt1 | gemini_runnable | output_parser

# Run the chain
final_output = sports_chain.invoke({"sport": "Cricket"})

print(final_output)

# ----------------
# Sequential Chain
# ----------------

# Sequential chains allow you to connect multiple chains and compose them into pipelines that execute some specific scenario.
# Returns only the final response.


In [ ]:
from google import genai
from langchain_core.messages import AIMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
import os
from dotenv import load_dotenv
import warnings
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
warnings.filterwarnings("ignore")

In [ ]:
# Gemini API key
env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")


# Step 1: Create the Gemini client
client = genai.Client(api_key=gemini_key)

In [ ]:
# Step 2: Wrap the Gemini API call as a LangChain Runnable
def call_gemini(prompt_value):
    # PromptTemplate produces a PromptValue object
    prompt_text = prompt_value.to_string()

    response = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt_text)

    # Return an AIMessage for StrOutputParser
    return AIMessage(content=response.text)

gemini_runnable = RunnableLambda(call_gemini)

In [ ]:
# Step 3: Content-generation prompt
content_prompt = PromptTemplate(input_variables=["title"],
    template="""
You are an expert content writer.

Given a title, create professional, high-quality content with a suitable heading.
The content should not exceed 1,000 words.

Title:
{title}

Content:
"""
)

# Step 4: Summary prompt
summary_prompt = PromptTemplate(
    input_variables=["contents"],
    template="""
You are an expert editor of text documents.

Given the content below, write a crisp 75-word summary.
Retain the same heading used in the original content.

Topic Content:
{contents}

Summary:
"""
)

# Step 5: Create the output parser
parser = StrOutputParser()

# Step 6: Create individual chains
content_chain = content_prompt | gemini_runnable | parser
summary_chain = summary_prompt | gemini_runnable | parser

# Step 7: Convert content output into summary input
prepare_summary_input = RunnableLambda(lambda contents: {"contents": contents})

# Step 8: Create the complete pipeline
chain = content_chain | prepare_summary_input | summary_chain

In [ ]:
# Step 9: Invoke the pipeline
qry = """ Should the usage of mobile phones be restricted for children under the age of 15 in India? """
response = chain.invoke({"title": qry})

In [ ]:
response.split()

In [ ]:
# To preserve all the intermediate results, you can use the RunnablePassthrough class to pass the output of one step to the next without any modification. 
# This allows you to inspect the intermediate results at each step of the pipeline.

In [ ]:
# Step 6: Create individual chains
content_chain = content_prompt | gemini_runnable | parser
summary_chain = summary_prompt | gemini_runnable | parser


# Generate content and preserve it in "contents"
# Then generate and add the summary
chain = (
    RunnablePassthrough.assign(contents=content_chain)
    | RunnablePassthrough.assign(summary=summary_chain)
)

In [ ]:
qry = """ Is Artificial Intelligence a threat or a boon to humanity? """
response = chain.invoke({"title": qry})

In [ ]:
response["title"]

In [ ]:
print("CONTENT:")
print(response["contents"])

In [ ]:
print("\nSUMMARY:")
print(response["summary"])